# End-to-End WCE Analysis Pipeline

A three-stage automated analysis system for **Wireless Capsule Endoscopy (WCE)** video footage.
A raw video is passed through the full pipeline and a clinical summary is produced — no manual frame review required.

```
Raw WCE Video
      │
      ▼
┌─────────────────────────────────────────────┐
│  Stage 1 — KMeans Video Summarisation       │
│  Stream BGR histograms → KMeans clustering  │
│  Output: ~15% of frames as keyframes        │
└─────────────────────┬───────────────────────┘
                      │ keyframes
                      ▼
┌─────────────────────────────────────────────┐
│  Stage 2 — GI Pathology Classification      │
│  Tri-backbone ensemble (VGG-16 + ResNet-50  │
│  + MobileNetV2) → 8-class softmax           │
└─────────────────────┬───────────────────────┘
                      │ polyp-related frames only
                      ▼
┌─────────────────────────────────────────────┐
│  Stage 3 — Polyp Segmentation               │
│  U-Net + DenseNet-169 encoder               │
│  Output: pixel-level binary mask            │
└─────────────────────────────────────────────┘
```

**Polyp-triggering classes:** `polyps`, `dyed-lifted-polyps`, `dyed-resection-margins`  
All other findings (esophagitis, normal anatomy) skip Stage 3.

In [ ]:
import os
import shutil
import numpy as np
import cv2
from pathlib import Path
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, concatenate,
    Input, LayerNormalization, Lambda, Conv2D
)
from tensorflow.keras.models import Model

%env SM_FRAMEWORK=tf.keras
import segmentation_models
from segmentation_models import Unet

%matplotlib inline

## Configuration

Set `VIDEO_PATH` and verify the model weight paths before running.

In [ ]:
# --- Input ---
VIDEO_PATH = "Dataset/hyperkvasir/0220d11b-ab12-4b02-93ce-5d7c205c7043.avi"   # <-- set this

# --- Stage 1: Video Summarisation ---
SAMPLING_RATE = 5     # use every Nth frame
SUMMARY_PCT   = 50    # % of sampled frames to keep as keyframes
NUM_BINS      = 32    # histogram bins per BGR channel
OUTPUT_DIR    = "keyframes"

# --- Stage 2: Classification ---
CLASSIFIER_WEIGHTS = "anomaly_classification\model_classification_20epochs_adam_lr0.01.h5"

CLASS_NAMES = [
    "dyed-lifted-polyps",
    "dyed-resection-margins",
    "esophagitis",
    "normal-cecum",
    "normal-pylorus",
    "normal-z-line",
    "polyps",
    "ulcerative-colitis",
]

# --- Stage 3: Segmentation ---
SEGMENTATION_WEIGHTS = "polyp_segmentation\model_segmentation_30epochs_customLoss.h5"
SEG_THRESHOLD        = 0.5

# Class indices (into CLASS_NAMES) that trigger segmentation
POLYP_CLASS_INDICES = {0, 1, 6}   # dyed-lifted-polyps, dyed-resection-margins, polyps

---
## Stage 1 — KMeans Video Summarisation

The video is never fully loaded into RAM. A two-pass streaming strategy is used:
- **Pass 1**: stream frames, compute BGR colour histograms, discard each frame immediately
- **KMeans**: cluster the (N × 48) histogram matrix
- **Pass 2**: seek to selected frame indices and decode only the keyframes

In [ ]:
def stream_histograms(video_path, sampling_rate=5, num_bins=16):
    """Single-pass: stream video, compute histograms, discard frames."""
    video_path = str(Path(video_path))
    if not Path(video_path).is_file():
        raise FileNotFoundError(f"Video not found: {video_path}")

    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    print(f"  {os.path.basename(video_path)}: {total} frames @ {fps:.1f} fps")

    hists, sampled_indices = [], []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % sampling_rate == 0:
            feat = np.concatenate([
                cv2.calcHist([frame], [c], None, [num_bins], [0, 256]).flatten()
                for c in range(3)
            ])
            hists.append(feat)
            sampled_indices.append(frame_idx)
        frame_idx += 1

    cap.release()
    hist_matrix = np.stack(hists).astype(np.float32)
    print(f"  Histogram matrix: {hist_matrix.shape}  ({hist_matrix.nbytes / 1e6:.1f} MB)")
    return hist_matrix, sampled_indices


def cluster_and_select(hist_matrix, sampled_indices, summary_pct=15):
    """KMeans on histogram matrix; return video frame indices of cluster representatives."""
    n_clusters = max(1, int(summary_pct * len(sampled_indices) / 100))
    print(f"  KMeans: {len(sampled_indices)} samples → {n_clusters} clusters")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(hist_matrix)
    dist = kmeans.transform(hist_matrix)
    best = sorted(set(int(np.argmin(dist[:, c])) for c in range(n_clusters)))
    keyframe_video_indices = [sampled_indices[i] for i in best]
    print(f"  Selected {len(keyframe_video_indices)} keyframes")
    return keyframe_video_indices


def extract_keyframes(video_path, keyframe_video_indices):
    """Pass 2: seek to selected indices and decode only those frames."""
    video_path = str(Path(video_path))
    cap = cv2.VideoCapture(video_path)
    keyframes = []
    for vid_idx in keyframe_video_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, vid_idx)
        ret, frame = cap.read()
        if ret:
            keyframes.append(frame)
    cap.release()
    return keyframes


def save_keyframes(keyframes, keyframe_indices, output_dir):
    out = Path(output_dir)
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    for i, (frame, vid_idx) in enumerate(zip(keyframes, keyframe_indices)):
        cv2.imwrite(str(out / f"keyframe_{i:03d}_f{vid_idx}.jpg"), frame)

In [ ]:
print("=" * 55)
print("STAGE 1 — KMeans Video Summarisation")
print("=" * 55)

hist_matrix, sampled_indices = stream_histograms(VIDEO_PATH, SAMPLING_RATE, NUM_BINS)
keyframe_video_indices = cluster_and_select(hist_matrix, sampled_indices, SUMMARY_PCT)
keyframes = extract_keyframes(VIDEO_PATH, keyframe_video_indices)
save_keyframes(keyframes, keyframe_video_indices, OUTPUT_DIR)

# Display keyframe grid
cols  = 4
n     = len(keyframes)
rows  = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
axes = np.array(axes).flatten()
for i, (frame, vid_idx) in enumerate(zip(keyframes, keyframe_video_indices)):
    axes[i].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Frame {vid_idx}", fontsize=9)
    axes[i].axis("off")
for j in range(n, len(axes)):
    axes[j].set_visible(False)
plt.suptitle(f"Stage 1 — {n} Keyframes Extracted", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Stage 2 — GI Pathology Classification

Each keyframe is classified into one of 8 GI findings using a **tri-backbone ensemble**  
(VGG-16 + ResNet-50 + MobileNetV2) with a shared SVD-whitened classification head.

Frames predicted as `polyps`, `dyed-lifted-polyps`, or `dyed-resection-margins` are forwarded to Stage 3.

In [ ]:
def build_classifier(weights_path):
    """Rebuild the tri-backbone ensemble and load trained weights."""
    def svd_whiten(x):
        s, u, v = tf.linalg.svd(x, compute_uv=True)
        return u @ tf.linalg.diag(s) @ tf.transpose(v)

    inp         = Input(shape=(224, 224, 3))
    base_vgg    = VGG16(include_top=False,       weights="imagenet", input_shape=(224, 224, 3))
    base_resnet = ResNet50(include_top=False,    weights="imagenet", input_shape=(224, 224, 3))
    base_mobile = MobileNetV2(include_top=False, weights="imagenet", input_shape=(224, 224, 3))

    out1 = Dense(256, activation="relu")(GlobalAveragePooling2D()(base_vgg(inp)))
    out2 = Dense(256, activation="relu")(GlobalAveragePooling2D()(base_resnet(inp)))
    out3 = Dense(256, activation="relu")(GlobalAveragePooling2D()(base_mobile(inp)))

    x = concatenate([out1, out2, out3])
    x = LayerNormalization(axis=1, epsilon=0.001)(x)
    x = Lambda(svd_whiten)(x)
    x = Dense(1024, activation="relu")(x)
    x = Dense(512,  activation="relu")(x)
    x = Dense(128,  activation="relu")(x)
    predictions = Dense(8, activation="softmax")(x)

    model = Model(inputs=inp, outputs=predictions)
    model.load_weights(weights_path)
    print(f"  Classifier loaded  ← '{weights_path}'")
    return model


def preprocess_for_classifier(frame):
    img = cv2.resize(frame, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0


print("Loading classification model...")
classifier = build_classifier(CLASSIFIER_WEIGHTS)

In [ ]:
print("=" * 55)
print("STAGE 2 — GI Pathology Classification")
print("=" * 55)

batch = np.stack([preprocess_for_classifier(f) for f in keyframes])
probs = classifier.predict(batch, verbose=0)
pred_indices = np.argmax(probs, axis=1)
pred_classes = [CLASS_NAMES[i] for i in pred_indices]
confidences  = probs[np.arange(len(probs)), pred_indices]

POLYP_COLOR  = "#e74c3c"
NORMAL_COLOR = "#27ae60"

cols  = 4
n     = len(keyframes)
rows  = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
axes = np.array(axes).flatten()

for i, (frame, vid_idx, cls, conf) in enumerate(
    zip(keyframes, keyframe_video_indices, pred_classes, confidences)
):
    color = POLYP_COLOR if pred_indices[i] in POLYP_CLASS_INDICES else NORMAL_COLOR
    axes[i].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Frame {vid_idx}\n{cls}\n{conf:.1%}", fontsize=8, color=color)
    axes[i].axis("off")
for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Stage 2 — GI Pathology Classification  (red = polyp-related)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Collect frames flagged for segmentation
polyp_frames = [
    (i, keyframes[i], keyframe_video_indices[i])
    for i in range(n)
    if pred_indices[i] in POLYP_CLASS_INDICES
]
print(f"\n{len(polyp_frames)} keyframe(s) flagged for polyp segmentation.")

---
## Stage 3 — Polyp Segmentation

Only frames classified as polyp-related reach this stage.  
A **U-Net with DenseNet-169 encoder** (pretrained on ImageNet) produces a pixel-level binary mask.  
The overlay highlights detected polyp regions in green.

In [ ]:
def build_segmentation_model(weights_path):
    """Rebuild U-Net + DenseNet-169 and load trained weights."""
    base = Unet(backbone_name="densenet169", encoder_weights="imagenet")
    inp  = Input(shape=(512, 512, 3))
    l1   = Conv2D(3, (1, 1))(inp)
    out  = base(l1)
    model = Model(inp, out, name=base.name)
    model.load_weights(weights_path)
    print(f"  Segmentation model loaded  ← '{weights_path}'")
    return model


seg_preprocess = segmentation_models.get_preprocessing("densenet169")

print("Loading segmentation model...")
seg_model = build_segmentation_model(SEGMENTATION_WEIGHTS)

In [ ]:
print("=" * 55)
print("STAGE 3 — Polyp Segmentation")
print("=" * 55)

seg_results = []

if not polyp_frames:
    print("  No polyp-related frames detected — segmentation skipped.")
else:
    for frame_idx, frame, vid_idx in polyp_frames:
        img_resized = cv2.resize(frame, (512, 512))
        img_rgb     = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
        inp_tensor  = seg_preprocess(img_rgb.astype(np.float32))
        inp_tensor  = np.expand_dims(inp_tensor, axis=0)

        mask = (seg_model.predict(inp_tensor, verbose=0)[0, ..., 0] > SEG_THRESHOLD).astype(np.uint8)

        overlay = img_rgb.copy()
        overlay[mask == 1] = (
            overlay[mask == 1] * 0.4 + np.array([0, 220, 0]) * 0.6
        ).astype(np.uint8)

        seg_results.append((frame_idx, vid_idx, img_rgb, mask, overlay))

    fig, axes = plt.subplots(len(seg_results), 3, figsize=(14, 4 * len(seg_results)))
    if len(seg_results) == 1:
        axes = axes[np.newaxis, :]

    for row, (frame_idx, vid_idx, img_rgb, mask, overlay) in enumerate(seg_results):
        cls_name = CLASS_NAMES[pred_indices[frame_idx]]
        axes[row, 0].imshow(img_rgb)
        axes[row, 0].set_title(f"Frame {vid_idx} — {cls_name}", fontsize=9)
        axes[row, 0].axis("off")
        axes[row, 1].imshow(mask, cmap="gray")
        axes[row, 1].set_title("Predicted Mask", fontsize=9)
        axes[row, 1].axis("off")
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title("Overlay (green = polyp)", fontsize=9)
        axes[row, 2].axis("off")

    plt.suptitle("Stage 3 — Polyp Segmentation", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Clinical Summary

In [ ]:
print("=" * 62)
print("CLINICAL SUMMARY")
print("=" * 62)
print(f"  Video      : {os.path.basename(VIDEO_PATH)}")
print(f"  Keyframes  : {len(keyframes)}  (sampled every {SAMPLING_RATE} frames, top {SUMMARY_PCT}%)")
print()
print(f"  {'Frame':>7}  {'Finding':<25}  {'Confidence':>10}  {'Segmented'}")
print("  " + "-" * 58)
for i, (vid_idx, cls, conf) in enumerate(
    zip(keyframe_video_indices, pred_classes, confidences)
):
    seg_tag = "  YES" if pred_indices[i] in POLYP_CLASS_INDICES else ""
    print(f"  {vid_idx:>7}  {cls:<25}  {conf:>10.1%}  {seg_tag}")
print("  " + "-" * 58)
print(f"  Polyp-related findings : {len(polyp_frames)} / {len(keyframes)} keyframes")
print(f"  Segmentation masks     : {len(seg_results)}")
print("=" * 62)